In [ ]:
%pip install scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [11]:
# กำหนดให้โชว์สูงสุดแค่ 10 แถว (ถ้าเกินจะขึ้น ... ตรงกลาง)
pd.set_option('display.max_rows', 10)

# กำหนดให้โชว์คอลัมน์สูงสุดแค่ 5 คอลัมน์ (ถ้าเกินจะซ่อนคอลัมน์กลางๆ)
pd.set_option('display.max_columns',None)

In [ ]:
# 1. โหลดข้อมูล (ใช้ r หน้า path เพื่อกัน Error เรื่องเครื่องหมาย \)
file_path = r'C:\Users\KS\Desktop\ยาเส้น1.xlsx'

try:
    df = pd.read_excel(file_path)
    print("--- โหลดข้อมูลสำเร็จ ---")
except FileNotFoundError:
    print("--- หาไฟล์ไม่เจอ! เช็ก Path หรือชื่อไฟล์อีกครั้ง ---")

--- โหลดข้อมูลสำเร็จ ---


In [ ]:




# 2. ตั้งค่าให้ Jupyter โชว์ข้อมูลทั้งหมด (ตามที่ต้องการ)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# 3. ตรวจสอบ Outlier ด้วยวิธี IQR (สถิติ)
# สมมติคอลัมน์ชื่อ 'น้ำหนัก_เข้า' (เปลี่ยนชื่อให้ตรงกับไฟล์คุณ)
col_target = 'เพิ่ม' 

Q1 = df[col_target].quantile(0.25)
Q3 = df[col_target].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# สร้าง DataFrame สำหรับรายการที่ "โดด" ผิดปกติ
outliers_iqr = df[(df[col_target] < lower_bound) | (df[col_target] > upper_bound)].copy()

# 4. ตรวจสอบความผิดปกติด้วย Logic (Yield % เพี้ยน)
# สมมติมีคอลัมน์ 'น้ำหนัก_ออก'
if 'ลด ' in df.columns:
    df['yield_pct'] = (df['ลด '] / df[col_target]) * 100
    # กรองรายการที่ Yield เกิน 100% (เป็นไปไม่ได้) หรือน้อยกว่า 60% (ผิดปกติมาก)
    logic_errors = df[(df['yield_pct'] > 100) | (df['yield_pct'] < 60)].copy()
else:
    logic_errors = pd.DataFrame()

# 5. แสดงผลลัพธ์
print(f"\n[สรุปการตรวจสอบ]")
print(f"- จำนวนข้อมูลทั้งหมด: {len(df)} รายการ")
print(f"- พบ Outlier ทางสถิติ (IQR): {len(outliers_iqr)} รายการ")
print(f"- พบรายการที่ Yield ผิดปกติ: {len(logic_errors)} รายการ")

print("\n--- รายการที่ควรตรวจสอบ (Outliers) ---")
display(outliers_iqr)

# 6. พล็อตกราฟดูการกระจายตัว
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
sns.boxplot(x=df[col_target])
plt.title('Boxplot: Check Outliers')

plt.subplot(1, 2, 2)
plt.scatter(df.index, df[col_target], alpha=0.5)
plt.axhline(upper_bound, color='r', linestyle='--', label='Upper Bound')
plt.axhline(lower_bound, color='r', linestyle='--', label='Lower Bound')
plt.title('Scatter: Distribution')
plt.legend()
plt.show()

In [ ]:
df.columns

Index(['คลัง ', '11', ' ตำแหน่งเก็บ', 'เพิ่ม', 'ลด ', 'คงเหลือ ', 'yield_pct'], dtype='object')

In [12]:
df[['ลด ' , 'เพิ่ม']].astype('float64')

,ลด,เพิ่ม
0,0.000,1.051
1,0.084,0.000
2,0.018,0.000
3,0.000,2.000
4,0.069,0.000
...,...,...
163,0.000,3.000
164,0.095,0.000
165,0.051,0.000
166,0.032,0.000


In [33]:

def splitnum( row ):
    target_cols = ['เพิ่ม', 'ลด '] # รายชื่อคอลัมน์ที่ต้องการแยก
    
    for col in target_cols:
        if col in df.columns:
           row[f'{col}_หน้า'], row[f'{col}_หลัง'] = str(row[col]).split('.') if '.' in str(row[col]) else (str(row[col], '0' ))
    return row
            

In [35]:
df = df.apply(splitnum , axis=1)

In [36]:
df.columns

Index(['คลัง ', '11', ' ตำแหน่งเก็บ', 'เพิ่ม', 'ลด ', 'คงเหลือ ', 'yield_pct',
       'เพิ่ม_หน้า', 'เพิ่ม_หลัง', 'ลด _หน้า', 'ลด _หลัง'],
      dtype='object')

In [15]:
df[['เพิ่ม_หน้า', 'ลด _หน้า']]*150

,เพิ่ม_หน้า,ลด _หน้า
0,150.0,0.0
1,0.0,0.0
2,0.0,0.0
3,300.0,0.0
4,0.0,0.0
...,...,...
163,450.0,0.0
164,0.0,0.0
165,0.0,0.0
166,0.0,0.0
